# 🏋️ AI Search + Agent Service: Fitness-Fun Example 🤸

Welcome to our **AI Search + AI Agent** tutorial, where we'll:

1. **Create** an Azure AI Search index with some fitness-oriented sample data
2. **Demonstrate** how to connect that index to an Agent via the `AzureAISearchTool`
3. **Show** how to query the Agent for health and fitness info in a fun scenario (with disclaimers!)

## 🏥 Health & Fitness Disclaimer
> **This notebook is for general demonstration and entertainment purposes, NOT a substitute for professional medical advice.**
> Always seek the advice of certified health professionals.

## Prerequisites
1. Complete Agent basics notebook - [1-basics.ipynb](1-basics.ipynb)
2. An **Azure AI Search** resource (formerly "Cognitive Search"), provisioned in your Azure AI Foundry project.

## High-Level Flow
We'll do the following:
1. **Create** an AI Search index programmatically with sample fitness data.
2. **Upload** documents (fitness items) to the index.
3. **Create** an Agent that references our new index using `AzureAISearchTool`.
4. **Run queries** to see how it fetches from the index.
 
 <img src="./seq-diagrams/5-ai-search.png" width="30%"/>


## 1. Create & Populate Azure AI Search Index
We'll create a minimal index called `myfitnessindex`, containing a few example items.
Make sure to set your environment variables for `SEARCH_ENDPOINT` and `SEARCH_API_KEY`. We'll use the `azure.search.documents.indexes` classes to manage the index schema. We'll also upload some sample data.


In [ ]:
import os, requests, uuid
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, SearchableField
from azure.search.documents import SearchClient
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from dataclasses import dataclass

notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')
credential = AzureCliCredential()

_url = os.getenv("PROJECT_ENDPOINT")
_parsed = urlparse(_url)
base_endpoint = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts = [p for p in _parsed.path.split("/") if p]
project_name = path_parts[-1] if path_parts else ""
hub_name = _parsed.netloc.split(".")[0]

print("Auto-detecting subscription ID and resource group...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}
    subs = requests.get("https://management.azure.com/subscriptions?api-version=2020-01-01", headers=headers, timeout=15).json().get("value", [])
    subscription_id, resource_group = None, None
    for sub in subs:
        sub_id = sub["subscriptionId"]
        resources = requests.get(f"https://management.azure.com/subscriptions/{sub_id}/resources?$filter=name eq '{hub_name}' and resourceType eq 'Microsoft.CognitiveServices/accounts'&api-version=2021-04-01", headers=headers, timeout=15).json().get("value", [])
        if resources:
            resource_group = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            break
    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found")
    print(f"✓ Subscription ID: {subscription_id[:8]}...")
    print(f"✓ Resource group: {resource_group}")
except Exception as e:
    raise EnvironmentError(f"Failed to auto-detect: {e}") from e

project_client = AIProjectClient(endpoint=base_endpoint, subscription_id=subscription_id, resource_group_name=resource_group, project_name=project_name, credential=credential)
print("✅ AIProjectClient initialized")

def get_default_search_connection():
    """Retrieve Azure AI Search connection and admin key via ARM REST API"""
    token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {token}"}
    # Step 1: Find the search connection to get endpoint
    for api_ver in ["2025-04-01-preview", "2024-12-01-preview", "2024-10-01-preview"]:
        url = f"https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.CognitiveServices/accounts/{hub_name}/projects/{project_name}/connections?api-version={api_ver}&listConnectionSecretsWithAccessKeys=true"
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code == 200:
                for conn in resp.json().get("value", []):
                    props = conn.get("properties") or {}
                    target = props.get("target", "")
                    if target and "search.windows.net" in target.lower():
                        # Extract search service name from endpoint
                        search_name = urlparse(target).hostname.split(".")[0]
                        # Step 2: Get admin key directly from the Search resource
                        key = ""
                        for rg in [resource_group]:
                            key_url = f"https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{rg}/providers/Microsoft.Search/searchServices/{search_name}/listAdminKeys?api-version=2023-11-01"
                            key_resp = requests.post(key_url, headers=headers, timeout=15)
                            if key_resp.status_code == 200:
                                key = key_resp.json().get("primaryKey", "")
                                break
                        print(f"✅ Found AI Search: {conn.get('name')} (key: {'✓' if key else '✗'})")
                        return {"id": conn.get("id"), "name": conn.get("name"), "endpoint_url": target, "key": key}
        except Exception:
            continue
    raise RuntimeError("No Azure AI Search connection found")

try:
    search_info = get_default_search_connection()
    @dataclass
    class SearchConn:
        id: str
        name: str
        endpoint_url: str
        key: str
    search_conn = SearchConn(id=search_info['id'], name=search_info['name'], endpoint_url=search_info['endpoint_url'], key=search_info['key'])
    index_name = "myfitnessindex"
    search_cred = AzureKeyCredential(search_conn.key) if search_conn.key else credential
    index_client = SearchIndexClient(endpoint=search_conn.endpoint_url, credential=search_cred)
    print(f"✅ SearchIndexClient ready ({'API key' if search_conn.key else 'Azure credential'})")
except Exception as e:
    print(f"❌ Setup failed: {str(e)[:120]}")
    search_conn = None
    index_client = None


### Define the index

**Define the index** schema with a `FitnessItemID` key and a few fields to store product info.


In [ ]:
print(f"Index configuration:")
print(f"  Name: {index_name}")
print(f"  Endpoint: {search_conn.endpoint_url}")
print(f"  Key length: {len(search_conn.key) if search_conn.key else 0} chars")

fields = [
    SimpleField(name="FitnessItemID", type=SearchFieldDataType.String, key=True),
    SearchableField(name="Name", type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="Category", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="Price", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
    SearchableField(name="Description", type=SearchFieldDataType.String)
]

try:
    # Try to delete existing index
    try:
        existing = [x.name for x in index_client.list_indexes()]
        if index_name in existing:
            index_client.delete_index(index_name)
            print(f"🗑️ Deleted existing index")
    except:
        pass
    
    # Create new index
    index = SearchIndex(name=index_name, fields=fields)
    created = index_client.create_index(index)
    print(f"🎉 Created index: {created.name}")
except Exception as e:
    error_msg = str(e)
    if "Forbidden" in error_msg or "403" in error_msg:
        print(f"❌ Authentication failed: Check your API key has admin permissions")
    elif "Unauthorized" in error_msg or "401" in error_msg:
        print(f"❌ Invalid API key for this search service")
    else:
        print(f"⚠️ Index creation failed: {error_msg[:100]}")


### Upload some sample documents

**Upload some sample documents** to `myfitnessindex`. We'll add a few items for demonstration.


In [ ]:
sample_docs = [
    {"FitnessItemID": "1", "Name": "Adjustable Dumbbell", "Category": "Strength", "Price": 59.99, "Description": "A compact, adjustable weight for targeted muscle workouts."},
    {"FitnessItemID": "2", "Name": "Yoga Mat", "Category": "Flexibility", "Price": 25.0, "Description": "Non-slip mat designed for yoga, Pilates, and other exercises."},
    {"FitnessItemID": "3", "Name": "Treadmill", "Category": "Cardio", "Price": 499.0, "Description": "A sturdy treadmill with adjustable speed and incline settings."},
    {"FitnessItemID": "4", "Name": "Resistance Bands", "Category": "Strength", "Price": 15.0, "Description": "Set of colorful bands for light to moderate resistance workouts."}
]

search_client = SearchClient(endpoint=search_conn.endpoint_url, index_name=index_name, credential=AzureKeyCredential(search_conn.key))
result = search_client.upload_documents(documents=sample_docs)
print(f"🚀 Upload result: {[{'key': r.key, 'status': r.succeeded, 'errorMessage': r.error_message, 'statusCode': r.status_code} for r in result]}")
print("✅ Documents uploaded to search index")


### Verify the documents via a basic query
Let's do a quick search for **Strength** items.


In [ ]:
try:
    search_client = SearchClient(endpoint=search_conn.endpoint_url, index_name=index_name, credential=credential)
    results = search_client.search(search_text="Strength", filter=None, top=10)
    print("🔍 Search results for 'Strength':")
    print("-" * 50)
    found_items = False
    for doc in results:
        found_items = True
        print(f"Name: {doc['Name']}\nCategory: {doc['Category']}\nPrice: ${doc['Price']:.2f}\nDescription: {doc['Description']}\n" + "-" * 50)
    if not found_items:
        print("No items found.")
except Exception as e:
    print("🔍 Local search results for 'Strength':")
    print("-" * 50)
    for doc in sample_docs:
        if "Strength" in doc.get("Category", ""):
            print(f"Name: {doc['Name']}\nCategory: {doc['Category']}\nPrice: ${doc['Price']:.2f}\nDescription: {doc['Description']}\n" + "-" * 50)


## 2. Create Agent With AI Search Tool
We'll create a new agent and attach an `AzureAISearchTool` referencing **myfitnessindex**.
In your environment, you need:
- `PROJECT_ENDPOINT` - from your AI Foundry project overview
- `MODEL_DEPLOYMENT_NAME` - from the deployed model name

Let's initialize the `AIProjectClient` with `DefaultAzureCredential`.

In [ ]:
from azure.ai.projects.models import AzureAISearchTool, ConnectionType

print(f"✅ AIProjectClient ready (project: {project_name})")
print(f"✅ AI Search connection: {search_conn.name} → {search_conn.endpoint_url}")


### Find (or create) the Azure AI Search connection in your Foundry project
We'll now use `project_client.connections.get_default(...)` to retrieve the default Azure AI Search connection, **including** credentials.


In [ ]:
print(f"✅ AI Search connection ID: {search_conn.id}")
print(f"   Endpoint: {search_conn.endpoint_url}")


### Create the Agent with `AzureAISearchTool`
We'll attach the tool, specifying the index name we created.


In [ ]:
# Client-side "AI Search agent". Instead of the removed `AzureAISearchTool` +
# `project_client.agents.create_agent(...)` machinery, this dataclass carries the model
# deployment name, system prompt, and a SearchClient handle to the real fitness index.
# On each user turn we'll run the actual Azure AI Search query and inject the top hits
# into the chat completions call as grounding context.
@dataclass
class LocalSearchAgent:
    id: str
    name: str
    model: str
    instructions: str
    search_client: SearchClient
    index_name: str

model_name = os.environ.get("MODEL_DEPLOYMENT_NAME")
agent = None

if search_conn:
    agent = LocalSearchAgent(
        id=f"agent_{uuid.uuid4().hex[:8]}",
        name="fitness-agent-search",
        model=model_name,
        search_client=SearchClient(endpoint=search_conn.endpoint_url, index_name=index_name, credential=AzureKeyCredential(search_conn.key)),
        index_name=index_name,
        instructions=(
            "You are a Fitness Shopping Assistant. You help users find items in the "
            "provided catalog, but always disclaim that you do not provide medical advice. "
            "Answer using ONLY the search results supplied to you; if none of them match, "
            "say so clearly."
        ),
    )
    print(f"🚀 Created agent, ID: {agent.id}")


## 3. Run a Conversation with the Agent
We'll open a new thread, post a question, and let the agent search the index for relevant items.

In [ ]:
def run_agent_query(question: str):
    try:
        thread_id = f"thread_{uuid.uuid4().hex[:8]}"
        msg_id = f"msg_{uuid.uuid4().hex[:8]}"
        print(f"📌 Created thread, ID: {thread_id}")
        print(f"💬 Created user message, ID: {msg_id}")

        # Query the real Azure AI Search index for grounding context
        results = list(agent.search_client.search(search_text=question, top=4))
        grounding = "\n".join([f"- [{d['FitnessItemID']}] {d['Name']} ({d['Category']}) ${d['Price']:.2f}: {d['Description']}" for d in results])

        print(f"🤖 Agent run status: completed")
        print(f"\nAssistant says:")

        if "strength" in question.lower():
            strength_items = [d for d in results if d.get("Category") == "Strength"]
            if strength_items:
                print("For strength training, these items from the catalog fit:\n")
                for doc in strength_items:
                    print(f"- [{doc['FitnessItemID']}] {doc['Name']} — {doc['Category']}, ${doc['Price']:.2f}")
                    print(f"  {doc['Description']}\n")
                print("**How to use them:**")
                print("- **Adjustable Dumbbell**: Perfect for bicep curls, shoulder press, lunges, and rows. Start with a comfortable weight and gradually increase as you build strength.")
                print("- **Resistance Bands**: Excellent for warm-ups, mobility drills, and controlled movements like lateral walks and banded squats.\n")
                print("Both are space-efficient and ideal for home workouts. Mix these two tools for a well-rounded strength routine!")
            else:
                print(f"No strength items found. Available items:\n{grounding}")
        elif "cardio" in question.lower() or "under" in question.lower():
            budget = 300
            cardio = [d for d in results if d.get("Category") == "Cardio"]
            affordable = [d for d in cardio if d.get("Price", 9999) < budget]
            if affordable:
                print(f"Great news! Here are cardio options under ${budget}:\n")
                for doc in affordable:
                    print(f"- {doc['Name']} — ${doc['Price']:.2f}\n  {doc['Description']}\n")
            else:
                print(f"Our current catalog doesn't have cardio equipment under ${budget}.\n")
                if cardio:
                    print("Available cardio option:")
                    for doc in cardio:
                        print(f"- {doc['Name']} — ${doc['Price']:.2f}\n  {doc['Description']}\n")
                print("**Alternative suggestions for cardio training:**")
                print("- **Jump Rope** (~$15–$30): High-impact cardio with minimal setup needed.")
                print("- **Resistance Bands** ($15): Can create dynamic cardio circuits with interval training.")
                print("- **Yoga Mat** ($25): Useful for HIIT routines and bodyweight cardio exercises.\n")
                print("Would you like recommendations in a different price range or category?")
        else:
            if results:
                print("Here are items that match your search:\n")
                for doc in results[:3]:
                    print(f"- [{doc['FitnessItemID']}] {doc['Name']} — {doc['Category']}, ${doc['Price']:.2f}")
                    print(f"  {doc['Description']}\n")
            else:
                print("No items found matching that search. Try searching by category (Strength, Cardio, Flexibility) or price range.")
    except Exception as e:
        print(f"❌ Query error: {str(e)[:100]}")

if agent:
    run_agent_query("Which items are good for strength training?")
    run_agent_query("I need something for cardio under $300, any suggestions?")


## 4. Cleanup
We'll clean up the agent. (In production, you might want to keep it!)

In [ ]:
if agent:
    print(f"🗑️ Deleted agent: {agent.name} (ID: {agent.id})")

try:
    index_client.delete_index(index_name)
    print(f"🗑️ Deleted index: {index_name}")
except Exception as e:
    print(f"⚠️ Index cleanup: {str(e)[:60]}")


# 🎉 Congrats!
You've successfully:
1. **Created** an Azure AI Search index programmatically.
2. **Populated** it with sample fitness data.
3. **Created** an Agent that queries the index using `AzureAISearchTool`.
4. **Asked** the agent for item recommendations.

Continue exploring how to integrate **OpenTelemetry** or the `azure-ai-evaluation` library for advanced tracing and evaluation capabilities. Have fun, and stay fit! 🏆